# 04 · Causal battery (P5–P8)

Pre-registration: `Docs/PREREGISTRATION.md`, section 7 (frozen 2026-11-15). This notebook is run in **three separate stages**, chosen by `STAGE` in Cell 2:

| `STAGE` | Fold | What happens | Then |
|---|---|---|---|
| `"LOC"` | LOC | Pairs, exact checks, the battery on every model, control bands and thresholds, exploratory sites, the fit of the high-level models. Writes `reports/04_loc_choices.json` and `reports/LOC_CHOICES.md` | Copy `LOC_CHOICES.md` into `Docs/`, commit, push |
| `"CONF_A"` | CONF-A, **once** | Confirmation with every choice fixed: seed rule for d, c, λ1; sigmoid budget; IIA; rung decisions; four-outcome table; DAS if needed | Notebook 05 |
| `"EXT"` | EXT, **once, at the very end** | Performance of the frozen models and replication of the CONF-A quantities | Nothing. EXT is closed |

The fold rules are enforced in `sepsis_core.data.FoldGuard`: CONF-A cannot be loaded before `Docs/LOC_CHOICES.md` is committed; CONF-B belongs to notebook 05; EXT needs the completion markers of 04 and 05 and can be opened for one pass only. In smoke mode none of CONF-A, CONF-B or EXT is ever loaded: the dry run "confirms" on a second draw of LOC pairs.

Reads from Drive at run time, and stops with a clear message if they are missing: `reports/03_heads.json` (heads carried forward, per model) and `reports/03_alive.json` (study outcome of section 6.2).

Governance: pair tables and per-pair statistics go to Drive and are never displayed. Every printed number is an aggregate.

In [ ]:
# ============================ CELL 1 · BOOTSTRAP: clone the repo, import sepsis_core from it ============================
REPO_URL = "https://github.com/Myakey/sepsis-transformer-analysis.git"
REPO_REF = "main"                                  # branch, tag or commit hash. PIN A COMMIT HASH for any run you will report.
REPO_DIR = "/content/sepsis-transformer-analysis"

import os, sys, subprocess
sys.dont_write_bytecode = True
if "google.colab" in sys.modules and not os.path.exists("/content/drive/MyDrive"):      # Drive holds the data and the checkpoints
    from google.colab import drive
    drive.mount("/content/drive")

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"git {' '.join(args[:2])} failed: {r.stderr.strip()[-300:]}")
    return r.stdout.strip()

def _auth_url(url):
    tok = None                                     # private repo: Colab secret GITHUB_TOKEN (Notebook access on)
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = os.environ.get("GITHUB_TOKEN")
    return url.replace("https://", f"https://{tok}@") if tok else url

if not os.path.isdir(os.path.join(REPO_DIR, ".git")): _git("clone", "--quiet", _auth_url(REPO_URL), REPO_DIR)
else: _git("fetch", "--quiet", "--all", cwd=REPO_DIR)
_git("checkout", "--quiet", REPO_REF, cwd=REPO_DIR)
if REPO_REF in _git("branch", "--format=%(refname:short)", cwd=REPO_DIR).split(): _git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)        # never leave a token in the clone's config
CORE_COMMIT = _git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["SEPSIS_PROJECT_ROOT"] = REPO_DIR
os.environ["SEPSIS_CORE_COMMIT"] = CORE_COMMIT
for m in [k for k in sys.modules if k == "sepsis_core" or k.startswith("sepsis_core.")]: del sys.modules[m]
if os.path.join(REPO_DIR, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(REPO_DIR, "Notebooks", "v4"))

import importlib.util
_need = [p for p in ("mlflow", "lightgbm") if importlib.util.find_spec(p) is None]
if _need: subprocess.call([sys.executable, "-m", "pip", "install", "-q", *_need])

import json, time, gc
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, calibration, controls, runs, plots, tracking, attention, physionet, correlational, causal, explain
print("sepsis_core", sepsis_core.__version__, "@ commit", CORE_COMMIT[:12], f"({REPO_REF})", "| torch", torch.__version__)
import hashlib

In [ ]:
# ============================ CELL 2 · SWITCHES ============================
SMOKE_TEST = False
STAGE = "LOC"            # "LOC" -> "CONF_A" -> (after notebook 05) "EXT". One stage per run, top to bottom.
OPT = dict(logit_sites=True, exploratory_sites=True, loose_donors=True, single_epoch_primary=True, das="auto")   # das: "auto" = only if no hand-specified model beats the floor
IIA_FLOOR_CONSTANT = "prereg"          # "random_key" ONLY after that deviation is approved in Docs/DEVIATIONS.md (see the note in Cell 14)
EXACT_TOL = 1e-5                       # exact checks E1-E4 must hold to float precision (fp32 logits)
I_CONFIRM_THE_SINGLE_EXT_PASS = False  # STAGE "EXT" does nothing unless this is True

runs.setup(smoke=SMOKE_TEST)
if STAGE not in ("LOC", "CONF_A", "EXT"): raise ValueError("STAGE must be 'LOC', 'CONF_A' or 'EXT'")
if not data.FoldGuard("causal").prereg_frozen(): raise PermissionError("Docs/PREREGISTRATION.md and its .frozen stamp are not both in this commit.")
SEEDS = list(C.CFG.SEEDS)[: (2 if SMOKE_TEST else 5)]; PRIMARY = SEEDS[0]; ARMS = ("softmax", "sigmoid"); sv = correlational.seed_verdict
REP = runs.report_path

# ---- outputs of notebook 03, read from Drive ----
for f_ in ("03_heads.json", "03_alive.json"):
    if not os.path.exists(REP(f_)):
        raise FileNotFoundError(f"{REP(f_)} is missing. Run notebook 03 first" + (" with SMOKE_TEST = True" if SMOKE_TEST else "") + ": Cells 5-7 write 03_heads.json and Cell 9 writes 03_alive.json.")
HEADS03 = json.load(open(REP("03_heads.json"))); ALIVE03 = json.load(open(REP("03_alive.json")))

def have(name, which="best.pt"): return os.path.exists(os.path.join(runs.model_dir(name), which))
def spec(arm, seed, which="best.pt"): return dict(trained=(f"{arm}_s{seed}", which), init=(f"{arm}_s{seed}", "init.pt"), shuf=(f"{arm}_shuf_v3", "best.pt"))
def carried(name, which="best.pt"):
    """Boolean (L,H) mask of the heads carried forward from 03 that can show effects on other hours (layers >= 2), and the 03 entry."""
    try: e = HEADS03[name][which]
    except KeyError: raise FileNotFoundError(f"reports/03_heads.json has no entry for {name} ({which}). Run the screen of that model in notebook 03 first.")
    m = np.zeros((e["L"], e["H"]), bool)
    for t in e["top"]:
        if not t["layer1"]: m[t["layer"] - 1, t["head"] - 1] = True
    if not m.any(): m[1:] = True                       # all six carried heads are in layer 1: fall back to every head in layers >= 2
    return m, e
for seed in SEEDS:
    for arm in ARMS:
        for role, (nm, wh) in spec(arm, seed).items():
            if not have(nm, wh): raise FileNotFoundError(f"model '{nm}' ({wh}) is not in {runs.run_root()}/models. Run notebook 02 first.")
        carried(f"{arm}_s{seed}")
print(f"stage: {STAGE} | seeds: {SEEDS} | 03 says (6.2): {ALIVE03.get('study_outcome')} | heads carried forward found for {len(SEEDS) * len(ARMS)} models")

In [ ]:
# ============================ CELL 3 · FIT/VAL bundle (the FIT-fitted preprocessor) and the class boundaries from FIT rows ============================
B = runs.prepare_mimic(stage="causal")
Q = causal.fit_quantiles(B)
print(f"severity classes from FIT rows: routine <= {Q['sev_routine_max']:g} ({Q['share_fit_rows_routine']:.1%} of rows) | severe >= {Q['sev_severe_min']:g} ({Q['share_fit_rows_severe']:.1%} of rows)")
print(f"arousal index z(HR)+z(Resp), hours with both measured: low <= {Q['arousal_low_max']:.3f} | high >= {Q['arousal_high_min']:.3f}")

In [ ]:
# ============================ CELL 4 · HELPERS ============================
PARTS = dict(priority=("up", causal.STATS), S0=("up_S0", causal.STATS), down=("down", causal.STATS), arousal=("arousal", ("lam",)), placebo=("placebo", ("lam",)), sofa_index=("sofa_t", ("lam",)))

def open_fold(fold, seed, label, guard=None):
    fb = causal.prepare_fold(fold, B["pre"], guard=guard or data.FoldGuard("causal"), stage=("final_ext" if fold == "EXT" else "causal"))
    PAIRS, INFO, T = causal.all_pairs(fb, B["pre"].feat, Q, seed=seed); sur = causal.severity_surrogates(fb, T, PAIRS)
    cdir = os.path.join(runs.run_root(), "causal", label); os.makedirs(cdir, exist_ok=True)
    sig = hashlib.sha256(b"".join(np.ascontiguousarray(PAIRS[k][c]).tobytes() for k in sorted(PAIRS) for c in ("row", "k", "t", "treat", "sham", "sham2", "loose"))).hexdigest()
    sp = os.path.join(cdir, "pairs.sha256")
    if os.path.exists(sp) and open(sp).read().strip() != sig:
        raise RuntimeError(f"the pair tables of {label} differ from the ones its cached battery was run on. Check out the commit that produced the cache, or move {cdir} away and re-run.")
    open(sp, "w").write(sig)
    bat = causal.Battery(fb, PAIRS, sur, B["pre"].feat, cdir, loose=bool(OPT["loose_donors"] and label == "LOC"))
    print(f"[{label}] subjects {fb['n_subjects']:,} | stays {fb['n_stays']:,} | subjects per fold in the split table: {fb['counts']}")
    for k, i in INFO.items():
        print(f"   pairs '{k}': {i['bases']:,} bases from {i['stays']:,} stays ({i['subjects']:,} subjects) | yield {i['yield_share']:.1%} of {i['candidates_tried']:,} candidates tried | "
              f"with a second sham donor {i['with_second_sham']:,} | donor pools: treat {i['treat_pool_rows']:,} rows, sham {i['sham_pool_rows']:,} rows")
    return fb, bat, INFO

def run_all(bat, label):
    for seed in SEEDS:
        for arm in ARMS:
            for role, (nm, wh) in spec(arm, seed).items():
                raw = bat.raw(nm, wh)
                if not causal.exact_ok(raw["checks"], EXACT_TOL):
                    raise RuntimeError(f"[{label}] exact check violated for {nm} ({wh}): {raw['checks']}. E1-E4 must be 0 to float precision; this is a bug, not a result. Send me this line.")
    print(f"[{label}] battery complete for {len(SEEDS) * len(ARMS)} trained models and their controls; every exact check <= {EXACT_TOL:g}")

def effs(bat, arm, seed, which="best.pt"): return {role: bat.eff(nm, wh) for role, (nm, wh) in spec(arm, seed, which).items()}

def decisions(loc_bat, conf_bat=None, which="best.pt", seeds=None):
    out = {}
    for seed in (seeds or SEEDS):
        for arm in ARMS:
            loc = effs(loc_bat, arm, seed, which); conf = effs(conf_bat, arm, seed, which) if conf_bat is not None else None
            out[(arm, seed)] = {k: causal.decide(loc, conf, part, stats) for k, (part, stats) in PARTS.items()}
    return out

def show(dec, part, stats, title):
    print(title)
    for (arm, seed), d in dec.items():
        cells = []
        for s in stats:
            r = d[part][s]; g = lambda role: (r.get(role, {}).get("confirmed") if r["confirmed_stage"] else r.get(role, {}).get("pass_loc"))
            cells.append(f"{s}: thr {r['threshold']:.3f} | trained {g('trained')} / untrained {g('init')} / shuffled {g('shuf')}" + (f" | E {r.get('E')}" if r["confirmed_stage"] else ""))
        print(f"   {arm}_s{seed:<6d} " + "   ".join(cells))

## Stage LOC: exploration, fitting, every choice

These cells run in every stage. In the later stages they reload the cached LOC battery (no recomputation) because the confirmation needs the LOC thresholds and signs.

In [ ]:
# ============================ CELL 5 · LOC: fold and pairs ============================
if STAGE != "EXT":
    FB_L, BAT_L, INFO_L = open_fold("LOC", seed=C.CFG.SEED + 1000, label="LOC")

In [ ]:
# ============================ CELL 6 · LOC: exact checks (7.3) and the battery on every model ============================
if STAGE != "EXT":
    for arm in ARMS:
        m_ = runs.load_trained(f"{arm}_s{PRIMARY}", FB_L["store"]); chk = causal.check_patched_forward(m_, FB_L, BAT_L.P["up"]); del m_
        print(f"{arm}_s{PRIMARY}: re-stated attention forward vs the module, no intervention: max |dz| {chk['max_abs_dz']:.2e}, max |dlogit| {chk['max_abs_dlogit']:.2e}")
        if max(chk.values()) > EXACT_TOL: raise RuntimeError("the re-stated attention forward does not reproduce the module: S4 / S5 / S6 cannot be trusted. Send me this line.")
    run_all(BAT_L, "LOC")
    for arm in ARMS: print(f"   {arm}_s{PRIMARY} exact checks (max absolute violation):", BAT_L.raw(f"{arm}_s{PRIMARY}")["checks"])

In [ ]:
# ============================ CELL 7 · LOC: effects, control bands, thresholds, heads passing on LOC (7.4, 7.5) ============================
if STAGE != "EXT":
    DEC_L = decisions(BAT_L)
    show(DEC_L, "priority", causal.STATS, "do(Priority_k), site S1 (PRIMARY). Heads in layers >= 2 passing on LOC: |severe - sham| > max(band, 0.05) with the CI excluding 0")
    show(DEC_L, "S0", causal.STATS, "\nsite S0 (raw-input splice, decay recomputed), reported beside S1:")
    show(DEC_L, "arousal", ("lam",), "\ndo(Arousal_t): sharpness slope over the keys before t")
    show(DEC_L, "placebo", ("lam",), "\nplacebo arousal index (circularly shifted): must not gate sharpness")
    e_ = BAT_L.eff(f"softmax_s{PRIMARY}")["up"]
    print(f"\nsoftmax_s{PRIMARY}, query-side reference set (keys before k; a reference, not a noise floor): max |shift| {np.nanmax(np.abs(e_['cP']['mean'][1:])):.3f}, "
          f"max |slope| {np.nanmax(np.abs(e_['lamP']['mean'][1:])):.3f} | output effect of the splice on the t+1 logit: {e_['dlogit']['mean'][0]:+.3f} SD [{e_['dlogit']['lo'][0]:+.3f}, {e_['dlogit']['hi'][0]:+.3f}]")
    if OPT["loose_donors"]:
        raw_ = BAT_L.raw(f"softmax_s{PRIMARY}")
        if "up__S1_loose__d" in raw_:
            dl = causal.boot_mean(raw_["up__S1_loose__d"] - raw_["up__S1_sham__d"], BAT_L.P["up"]["subj"])[0]; dm = e_["d"]["mean"]
            print(f"plausibility: direct effect with loosely matched donors vs context-matched donors, correlation across heads {np.corrcoef(dl[1:].ravel(), dm[1:].ravel())[0, 1]:+.3f}, "
                  f"mean |d| {np.nanmean(np.abs(dl[1:])):.3f} vs {np.nanmean(np.abs(dm[1:])):.3f}")

In [ ]:
# ============================ CELL 8 · LOC: model-level sites S4 (logit delta) and S6 (query scaling), primary seed ============================
LOGIT_SITES = {}
if STAGE != "EXT" and OPT["logit_sites"]:
    for arm in ARMS:
        for wh in ("best.pt", "init.pt"):
            nm = f"{arm}_s{PRIMARY}"; m_ = runs.load_trained(nm, FB_L["store"], wh); r = causal.run_logit_sites(m_, FB_L, BAT_L.P["up"], n_max=(60 if SMOKE_TEST else 800)); del m_
            sd = BAT_L.raw(nm, wh)["logit_sd"]; g = BAT_L.P["up"]["subj"]; nL = sum(k.startswith("S4_L") and k.endswith("__dlogit") for k in r); rec = {}
            for l in range(nL):
                x = r[f"S4_L{l + 1}__dlogit"][:, 0] / sd[0]; mean, lo, hi = causal.boot_mean(x, g[: len(x)])
                oth = float(np.nanmean((r[f"S4_L{l + 1}__dM"] - r[f"S4_L{l + 1}__da_k"])[:, l]))
                rec[f"S4_L{l + 1}"] = dict(output_effect_sd=float(mean), lo=float(lo), hi=float(hi), mean_change_of_other_weights_same_layer=oth)
            for l in range(1, nL):
                for sc in C.CFG.S6_SCALES:
                    k = f"S6_L{l + 1}_x{sc}"; rec[k] = dict(row_entropy_change=float(np.nanmean(r[k + "__dH"][:, l])), slope=float(np.nanmean(r[k + "__lam"][:, l])),
                                                           output_effect_sd=float(np.nanmean(r[k + "__dlogit"][:, 0] / sd[0])))
            LOGIT_SITES[f"{nm}__{wh}"] = rec
            print(f"{nm} ({wh}) S4, +{C.CFG.S4_DELTA:g} on z_tk: t+1 logit moves by " + ", ".join(f"L{l + 1} {rec[f'S4_L{l + 1}']['output_effect_sd']:+.4f} SD" for l in range(nL)) +
                  " | other weights in that layer change by " + ", ".join(f"{rec[f'S4_L{l + 1}']['mean_change_of_other_weights_same_layer']:+.4f}" for l in range(nL)))
            print("      S6, query scaled (reference for a pure gain change): " + " | ".join(f"{k[3:]}: dH {v['row_entropy_change']:+.3f}, slope {v['slope']:+.3f}" for k, v in rec.items() if k.startswith("S6")))

In [ ]:
# ============================ CELL 9 · LOC, EXPLORATORY: patch sites S2 (residual sweep), S3 (attention vs MLP route), S5 (value vector) ============================
DAS_LAYER = {arm: 2 for arm in ARMS}; SITES = {}
if STAGE != "EXT" and OPT["exploratory_sites"]:
    for arm in ARMS:
        nm = f"{arm}_s{PRIMARY}"; m_ = runs.load_trained(nm, FB_L["store"]); s_ = causal.run_patch_sites(m_, FB_L, BAT_L.P["up"], n_max=(60 if SMOKE_TEST else 500)); del m_
        SITES[nm] = causal.restoration(s_, float(BAT_L.raw(nm)["logit_sd"][0])); re = SITES[nm]
        print(f"{nm} EXPLORATORY restoration of the t+1 logit ({re['pairs_used']} of {re['pairs_total']} pairs with a usable denominator), ratio of means:")
        for fam in ("S2", "S3_att", "S3_mlp", "S5"): print(f"   {fam:<7s} " + "  ".join(f"{k.split('_')[-1]} {v['ratio_of_means']:+.3f}" for k, v in re.items() if k.startswith(fam + "_L") and v["ratio_of_means"] is not None))
        cand = {int(k.split("_L")[1]): v["ratio_of_means"] for k, v in re.items() if k.startswith("S2_L") and int(k.split("_L")[1]) >= 2 and v["ratio_of_means"] is not None}
        if cand: DAS_LAYER[arm] = max(cand, key=cand.get)
    print("layer chosen for DAS (largest whole-component restoration among layers >= 2):", DAS_LAYER)

In [ ]:
# ============================ CELL 10 · LOC: fit of the high-level models (sign of log-priority per head, tertile cut points), IIA on LOC ============================
SGN, CUTS, IIA_L = {}, {}, {}
if STAGE != "EXT":
    for seed in SEEDS:
        for arm in ARMS:
            nm = f"{arm}_s{seed}"; raw = BAT_L.raw(nm); SGN[nm] = causal.beta_sign(raw); CUTS[nm] = causal.iia_risk(raw, BAT_L.P["up"])["cuts"]; hm, _ = carried(nm)
            IIA_L[nm] = {part: causal.iia_table(causal.iia(raw, BAT_L.P[part], SGN[nm], part, floor=IIA_FLOOR_CONSTANT), BAT_L.P[part], hm) for part in ("up", "arousal")}
    for arm in ARMS:
        t_ = IIA_L[f"{arm}_s{PRIMARY}"]
        print(f"{arm}_s{PRIMARY} on LOC (descriptive; the decision is on CONF-A), carried-forward heads: " +
              " | ".join(f"{M_} {t_['up'][M_]['iia']['mean']:.3f} (floor {t_['up'][M_]['floor']['mean']:.3f})" for M_ in ("M0", "M1", "M2")) +
              f" | sham pairs, no structure: {t_['up']['sham_pairs_no_change']['mean']:.3f} | do(Arousal): M2 {t_['arousal']['M2']['iia']['mean']:.3f} (floor {t_['arousal']['M2']['floor']['mean']:.3f})")

In [ ]:
# ============================ CELL 11 · LOC CHOICES: written in stage LOC, verified against the committed file afterwards ============================
CHOICES_JSON, CHOICES_MD = REP("04_loc_choices.json"), REP("LOC_CHOICES.md")
if STAGE == "LOC":
    payload = dict(core_commit=CORE_COMMIT, run=C.CFG.RUN_NAME, quantiles=Q, pairs=INFO_L, exact_tolerance=EXACT_TOL, iia_floor_constant_models=IIA_FLOOR_CONSTANT,
                   models={f"{a}_s{s}": d["priority"] for (a, s), d in DEC_L.items()}, arousal={f"{a}_s{s}": d["arousal"] for (a, s), d in DEC_L.items()},
                   beta_sign={n: v.tolist() for n, v in SGN.items()}, risk_cuts=CUTS, das_layer=DAS_LAYER, carried_heads={n: HEADS03[n]["best.pt"]["top"] for n in SGN},
                   logit_sites=LOGIT_SITES, exploratory_sites=SITES)
    cid = causal.write_loc_choices(payload, CHOICES_JSON, CHOICES_MD)
    print(f"LOC stage complete. Choices id {cid}")
    print(f"NEXT: 1) download {CHOICES_MD} and save it as Docs/LOC_CHOICES.md in the project folder; 2) commit and push; 3) set REPO_REF (Cell 1) to that commit and STAGE = 'CONF_A'; 4) run all.")
    print("Send me the printed output of Cells 3-10 before the CONF-A run if you want it checked first. Nothing from CONF-A has been loaded.")
elif STAGE == "CONF_A":
    CH = json.load(open(CHOICES_JSON)) if SMOKE_TEST else causal.verify_loc_choices(CHOICES_JSON)
    for (a, s), d in DEC_L.items():
        for st_ in causal.STATS:
            if abs(CH["models"][f"{a}_s{s}"][st_]["threshold"] - d["priority"][st_]["threshold"]) > 1e-9:
                raise RuntimeError(f"the LOC threshold of {a}_s{s} ({st_}) differs from the committed choice: the LOC battery changed after Docs/LOC_CHOICES.md was written.")
    SGN = {n: np.array(v, float) for n, v in CH["beta_sign"].items()}; CUTS = CH["risk_cuts"]; DAS_LAYER = {k: int(v) for k, v in CH["das_layer"].items()}
    print(f"committed LOC choices verified (id {CH['loc_choices_id']}): thresholds match, signs and cut points taken from the committed file")

## Stage CONF-A: confirmation, once, with every choice fixed

In [ ]:
# ============================ CELL 12 · CONF-A: fold, pairs, battery ============================
if STAGE == "CONF_A":
    FB_C, BAT_C, INFO_C = open_fold("LOC" if SMOKE_TEST else "CONF_A", seed=C.CFG.SEED + 2000, label=("LOC_as_confirmation_SMOKE" if SMOKE_TEST else "CONF_A"))
    if not SMOKE_TEST: print("CONF-A / CONF-B subject counts (append this line to Docs/PREREGISTRATION.frozen):", {k: v for k, v in FB_C["counts"].items() if k.startswith("CONF_")})
    run_all(BAT_C, "CONF-A"); DEC = decisions(BAT_L, BAT_C)

In [ ]:
# ============================ CELL 13 · CONF-A: seed rule for d, c and lambda1 (5.6, 7.5); sigmoid budget ============================
if STAGE == "CONF_A":
    def E_of(arm, part, stat, key="E"): return [DEC[(arm, s)][part][stat].get(key) for s in SEEDS]
    show(DEC, "priority", causal.STATS, "heads in layers >= 2 that pass on LOC AND are confirmed on CONF-A; E = trained - max(untrained, shuffled)")
    show(DEC, "arousal", ("lam",), "\ndo(Arousal_t):"); show(DEC, "placebo", ("lam",), "\nplacebo index:")
    VERD = {}
    for arm in ARMS:
        VERD[arm] = dict(d=sv(E_of(arm, "priority", "d")), c=sv(E_of(arm, "priority", "c")), c_negative=sv(E_of(arm, "priority", "c", "E_neg")),
                         lam=sv(E_of(arm, "priority", "lam")), lam_positive=sv(E_of(arm, "priority", "lam", "E_pos")),
                         lam_absent_grade=causal.absent_grade([DEC[(arm, s)]["priority"]["lam"] for s in SEEDS]),
                         S0_d=sv(E_of(arm, "S0", "d")), S0_lam=sv(E_of(arm, "S0", "lam")), down_d=sv(E_of(arm, "down", "d")),
                         arousal_lam=sv(E_of(arm, "arousal", "lam")), arousal_lam_positive=sv(E_of(arm, "arousal", "lam", "E_pos")),
                         arousal_absent_grade=causal.absent_grade([DEC[(arm, s)]["arousal"]["lam"] for s in SEEDS]),
                         placebo_lam=sv(E_of(arm, "placebo", "lam")), sofa_index_lam=sv(E_of(arm, "sofa_index", "lam")))
        v = VERD[arm]
        print(f"\n{arm}: direct effect d (manipulation check) {v['d']} | common shift c {v['c']} (negative: {v['c_negative']}) | selectivity slope {v['lam']} (positive: {v['lam_positive']})"
              + (f" [{v['lam_absent_grade']}]" if v["lam"] == "ABSENT" else ""))
        print(f"{' ' * len(arm)}  S0 beside S1: d {v['S0_d']}, slope {v['S0_lam']} | down pairs: d {v['down_d']} | do(Arousal) slope {v['arousal_lam']} (positive: {v['arousal_lam_positive']}) | "
              f"placebo index {v['placebo_lam']} | SOFA_t as the index (sensitivity) {v['sofa_index_lam']}")
    KAPPA = {arm: {s: causal.kappa(BAT_C.raw(f"{arm}_s{s}"), BAT_C.P["up"]) for s in SEEDS} for arm in ARMS}
    for arm in ARMS:
        print(f"\n{arm} compensation kappa = -(change of the other weights) / (change of the weight to k), layers >= 2, per seed: " +
              ", ".join(f"{k['kappa']:.2f} [{k['lo']:.2f}, {k['hi']:.2f}]" for k in KAPPA[arm].values()) + ("   (1 by construction under a softmax)" if arm == "softmax" else ""))
    BUDGET = dict(learned_budget=correlational.all_seeds([k["verdict"] == "learned budget" for k in KAPPA["sigmoid"].values()]),
                  independent_gates=correlational.all_seeds([k["verdict"] == "independent gates" for k in KAPPA["sigmoid"].values()]))
    print(f"sigmoid: learned budget (kappa CI inside [0.5, 1.5], 5 of 5 seeds): {BUDGET['learned_budget']} | independent gates (inside [-0.25, 0.25]): {BUDGET['independent_gates']}")

In [ ]:
# ============================ CELL 14 · CONF-A: IIA for M0 / M1 / M2 (7.7), interaction and dose-response (7.6), rung decisions (7.9) ============================
# NOTE on the floor. For a model that predicts a constant vector, the frozen floor (log P permuted within the set) cannot differ from the model's own
# IIA when there is no structure, so 'M0 beats the floor' cannot be met under that floor. Both the frozen floor and the proposed alternative
# (floor = 'random_key') are computed; the decision below uses IIA_FLOOR_CONSTANT, which stays "prereg" until the deviation is approved.
if STAGE == "CONF_A":
    IIA_C, IIA_ALT, INTER, DOSE, RISK = {}, {}, {}, {}, {}
    for seed in SEEDS:
        for arm in ARMS:
            nm = f"{arm}_s{seed}"; raw = BAT_C.raw(nm); hm, _ = carried(nm); P_ = BAT_C.P
            IIA_C[nm] = {part: causal.iia_table(causal.iia(raw, P_[part], SGN[nm], part, floor=IIA_FLOOR_CONSTANT), P_[part], hm) for part in ("up", "arousal")}
            IIA_ALT[nm] = causal.iia_table(causal.iia(raw, P_["up"], SGN[nm], "up", floor="random_key"), P_["up"], hm)
            RISK[nm] = causal.iia_risk(raw, P_["up"], cuts=CUTS[nm]); ge2 = np.ones_like(hm); ge2[0] = False
            INTER[nm] = causal.by_tertile(causal.pool_heads(raw["up__S1_treat__d"] - raw["up__S1_sham__d"], ge2), P_["up"]["arousal_t"], P_["up"]["subj"])
            DOSE[nm] = causal.by_tertile(causal.pool_heads(raw["arousal__A_treat__lam"] - raw["arousal__A_sham__lam"], ge2), P_["arousal"]["ar_treat"] - P_["arousal"]["ar_base"], P_["arousal"]["subj"])
    def every(fn, arm): return correlational.all_seeds([fn(f"{arm}_s{s}") for s in SEEDS])
    RUNG = {}
    for arm in ARMS:
        t_ = IIA_C[f"{arm}_s{PRIMARY}"]; v = VERD[arm]
        b = dict(M0_beats_floor=every(lambda n: causal.beats(IIA_C[n]["up"]["M0"]["minus_floor"]), arm), M1_beats_floor=every(lambda n: causal.beats(IIA_C[n]["up"]["M1"]["minus_floor"]), arm),
                 M2_beats_floor=every(lambda n: causal.beats(IIA_C[n]["arousal"]["M2"]["minus_floor"]), arm), M1_over_M0=every(lambda n: causal.beats(IIA_C[n]["up"]["M1_minus_M0"]), arm),
                 M2_over_M0_and_M1=every(lambda n: causal.beats(IIA_C[n]["arousal"]["M2_minus_M0"]) and causal.beats(IIA_C[n]["arousal"]["M2_minus_M1"]), arm),
                 positive_interaction=every(lambda n: bool(INTER[n].get("ci") and INTER[n]["ci"][0] > 0), arm),
                 M0_beats_floor_ALTERNATIVE_random_key=every(lambda n: causal.beats(IIA_ALT[n]["M0"]["minus_floor"]), arm))
        RUNG[arm] = dict(checks=b, rung1_M0=bool(v["d"] == "PRESENT" and v["lam"] == "ABSENT" and b["M0_beats_floor"] == "YES"),
                         rung2_M1=bool(v["lam_positive"] == "PRESENT" and b["M1_over_M0"] == "YES"),
                         rung3_M2=bool(v["arousal_lam_positive"] == "PRESENT" and v["placebo_lam"] == "ABSENT" and b["positive_interaction"] == "YES" and b["M2_over_M0_and_M1"] == "YES"))
        RUNG[arm]["evidence_against_arousal_gating"] = bool((not RUNG[arm]["rung3_M2"]) and ((v["arousal_lam"] == "ABSENT" and v["arousal_absent_grade"] == "equivalent to zero") or v["placebo_lam"] == "PRESENT"))
        print(f"\n{arm}_s{PRIMARY} IIA on CONF-A, carried-forward heads [mean, 95% CI]: " + " | ".join(f"{M_} {t_['up'][M_]['iia']['mean']:.3f} minus floor {t_['up'][M_]['minus_floor']['mean']:+.3f} [{t_['up'][M_]['minus_floor']['lo']:+.3f}, {t_['up'][M_]['minus_floor']['hi']:+.3f}]" for M_ in ("M0", "M1", "M2")))
        print(f"   M1 - M0 {t_['up']['M1_minus_M0']['mean']:+.3f} [{t_['up']['M1_minus_M0']['lo']:+.3f}, {t_['up']['M1_minus_M0']['hi']:+.3f}] | do(Arousal): M2 - M0 {t_['arousal']['M2_minus_M0']['mean']:+.3f} "
              f"[{t_['arousal']['M2_minus_M0']['lo']:+.3f}, {t_['arousal']['M2_minus_M0']['hi']:+.3f}] | sham pairs without structure {t_['up']['sham_pairs_no_change']['mean']:.3f} | "
              f"IIA_risk direction {RISK[f'{arm}_s{PRIMARY}']['direction']['mean']:.3f}, tertile agreement {RISK[f'{arm}_s{PRIMARY}']['tertile_agreement']:.3f}")
        print(f"   across seeds (5 of 5 needed): {b}")
        print(f"   interaction, d by natural arousal tertile of the readout hour: {INTER[f'{arm}_s{PRIMARY}'].get('means')} top-bottom CI {INTER[f'{arm}_s{PRIMARY}'].get('ci')} | "
              f"dose-response of the arousal slope by size of the arousal change: {DOSE[f'{arm}_s{PRIMARY}'].get('means')}")
        print(f"   RUNG 1 (M0, allocation by normalisation): {'SUPPORTED' if RUNG[arm]['rung1_M0'] else 'not supported'} | RUNG 2 (M1): {'SUPPORTED' if RUNG[arm]['rung2_M1'] else 'not supported'} | "
              f"RUNG 3 (M2, arousal gating): {'SUPPORTED' if RUNG[arm]['rung3_M2'] else 'not supported'} | evidence against arousal gating in this model: {RUNG[arm]['evidence_against_arousal_gating']}")

In [ ]:
# ============================ CELL 15 · THE FOUR-OUTCOME TABLE (section 8), read according to the study outcome of 6.2 ============================
if STAGE == "CONF_A":
    soft = VERD["softmax"]["lam"]; sig = "PRESENT" if "PRESENT" in (VERD["sigmoid"]["c_negative"], VERD["sigmoid"]["lam"]) else ("ABSENT" if VERD["sigmoid"]["c_negative"] == "ABSENT" and VERD["sigmoid"]["lam"] == "ABSENT" else "SEED-DEPENDENT")
    study = ALIVE03.get("study_outcome"); FOUR = dict(softmax=soft, sigmoid=sig, study_outcome_6_2=study)
    cells = {("PRESENT", "PRESENT"): "learned, normaliser-independent competition (check: sham and untrained controls clean in both arms)",
             ("PRESENT", "ABSENT"): "competition scaffolded by the zero-sum constraint (check: sigmoid alive, no drift)",
             ("ABSENT", "PRESENT"): "floor effect in softmax, or a different (gating) solution (check: floor fraction from 03, softmax on dz, sigmoid mass regression)",
             ("ABSENT", "ABSENT"): "no learned selective competition (M0); grade: softmax " + VERD["softmax"]["lam_absent_grade"] + ", sigmoid " + VERD["sigmoid"]["lam_absent_grade"]}
    print(f"softmax pattern (selectivity slope under do(Priority)): {soft} | sigmoid pattern (negative common shift or slope): {sig} | 6.2: {study}")
    if study != "sigmoid ALIVE":
        FOUR["cell"] = None
        print("NO CELL IS CLAIMED. " + ("Attention contributes little in either arm: the verdicts above are patterns in logits that the prediction does not depend on; rungs 1-3 are statements about the attention logits only."
                                        if study == "ATTENTION CONTRIBUTES LITTLE IN EITHER ARM" else "The sigmoid arm cannot serve as a null; the softmax verdict is reported alone."))
    elif "SEED-DEPENDENT" in (soft, sig) or "INCOMPLETE" in soft + sig:
        FOUR["cell"] = None; print("NO CELL IS ASSIGNED: unstable across seeds (or fewer than 5 seeds).")
    else:
        FOUR["cell"] = cells[(soft, sig)]; print("CELL:", FOUR["cell"])
    print("S3 trigger (sigmoid alive AND learned budget in 5 of 5 seeds AND d PRESENT in the sigmoid arm):", bool(study == "sigmoid ALIVE" and BUDGET["learned_budget"] == "YES" and VERD["sigmoid"]["d"] == "PRESENT"))

In [ ]:
# ============================ CELL 16 · DAS (7.8): only if no hand-specified model beats the floor ============================
# The match rule used by causal.das is an operationalisation the pre-registration did not fix (see its docstring). It is logged as a proposed
# deviation; treat the numbers below as provisional until that entry is approved in Docs/DEVIATIONS.md.
DAS = {}
if STAGE == "CONF_A":
    for arm in ARMS:
        none_beats = not any(RUNG[arm]["checks"][k] == "YES" for k in ("M0_beats_floor", "M1_beats_floor", "M2_beats_floor"))
        if not (OPT["das"] is True or (OPT["das"] == "auto" and none_beats)): print(f"{arm}: DAS not run (a hand-specified model beats the floor, or DAS is switched off)"); continue
        nm = f"{arm}_s{PRIMARY}"; L_ = int(DAS_LAYER.get(arm, 2)); args = (FB_L, BAT_L.P["up"], FB_C, BAT_C.P["up"], L_); nmax = 60 if SMOKE_TEST else None; g = BAT_C.P["up"]["subj"]
        mt = runs.load_trained(nm, FB_L["store"]); mi = runs.load_trained(nm, FB_L["store"], "init.pt")
        R_ = dict(trained=causal.das(mt, *args, n_max=nmax), shuffled_labels=causal.das(mt, *args, shuffle_labels=True, n_max=nmax),
                  random_subspace=causal.das(mt, *args, train=False, n_max=nmax), random_network=causal.das(mi, *args, n_max=nmax)); del mt, mi
        rec = {}
        for m_ in list(C.CFG.DAS_DIMS):
            d_ = {c_: causal.boot_mean(R_["trained"][m_] - R_[c_][m_], g[: len(R_["trained"][m_])]) for c_ in ("shuffled_labels", "random_subspace", "random_network")}
            rec[m_] = dict(iia=float(np.nanmean(R_["trained"][m_])), controls={c_: float(np.nanmean(R_[c_][m_])) for c_ in d_},
                           beats_all_three=bool(all(float(v[0]) >= C.CFG.IIA_MARGIN and float(v[1]) > 0 for v in d_.values())))
        whole = float(np.nanmean(R_["trained"]["whole"]))
        DAS[arm] = dict(layer=L_, by_dim=rec, whole_component=whole, counts=bool(any(r["beats_all_three"] and r["iia"] <= whole + C.CFG.IIA_MARGIN for r in rec.values())))
        print(f"{nm} DAS at the residual stream entering layer {L_}, hour k: whole-component IIA {whole:.3f}")
        for m_, r in rec.items(): print(f"   m = {m_:<2d} IIA {r['iia']:.3f} | controls {r['controls']} | beats all three by >= {C.CFG.IIA_MARGIN}: {r['beats_all_three']} | illusion flag (subspace above whole component): {r['iia'] > whole + C.CFG.IIA_MARGIN}")
        print(f"   DAS counts: {DAS[arm]['counts']}" + ("" if DAS[arm]["counts"] else "  -> the alignment is reported as absent (evidence against the hypothesis, PDF P8)"))

In [ ]:
# ============================ CELL 17 · SENSITIVITY: best single epoch instead of the soup, primary seed ============================
SINGLE = {}
if STAGE == "CONF_A" and OPT["single_epoch_primary"]:
    for arm in ARMS:
        nm = f"{arm}_s{PRIMARY}"
        if not have(nm, "best_epoch.pt"): continue
        for bat_, lab_ in ((BAT_L, "LOC"), (BAT_C, "CONF-A")):
            if not causal.exact_ok(bat_.raw(nm, "best_epoch.pt")["checks"], EXACT_TOL): raise RuntimeError(f"[{lab_}] exact check violated for {nm} (best_epoch.pt)")
        d = causal.decide(effs(BAT_L, arm, PRIMARY, "best_epoch.pt"), effs(BAT_C, arm, PRIMARY, "best_epoch.pt"), "up"); SINGLE[nm] = d
        print(f"{nm} best single epoch: " + " | ".join(f"{s}: E {d[s].get('E')} (soup: {DEC[(arm, PRIMARY)]['priority'][s].get('E')})" for s in causal.STATS))

In [ ]:
# ============================ CELL 18 · SAVE (this is what to send back) ============================
if STAGE == "CONF_A":
    OUT = dict(core_commit=CORE_COMMIT, run=C.CFG.RUN_NAME, seeds=SEEDS, smoke=SMOKE_TEST, loc_choices_id=CH.get("loc_choices_id"), quantiles=Q, pairs=dict(LOC=INFO_L, CONF_A=INFO_C),
               decisions={f"{a}_s{s}": d for (a, s), d in DEC.items()}, verdicts=VERD, kappa=KAPPA, budget=BUDGET, iia=IIA_C, iia_alternative_floor=IIA_ALT, iia_risk=RISK,
               interaction=INTER, dose_response=DOSE, rungs=RUNG, four_outcome=FOUR, das=DAS, single_epoch=SINGLE, alive_03=ALIVE03.get("study_outcome"),
               iia_floor_constant_models=IIA_FLOOR_CONSTANT, exact_tolerance=EXACT_TOL)
    runs.save_json(OUT, "04_summary.json")
    if not SMOKE_TEST: runs.save_json(dict(done=True, time=time.strftime("%Y-%m-%dT%H:%M:%S"), core_commit=CORE_COMMIT, loc_choices_id=CH.get("loc_choices_id")), "04_confA_done.json")
    print("CONF-A stage complete. CONF-A is not loaded again. Send back the printed output of Cells 12-17 or reports/04_summary.json (all aggregate).")

## Stage EXT: the single final pass (only after notebook 05 is complete)

Contents fixed in the pre-registration (section 3): (i) performance of the frozen models; (ii) replication of the CONF-A confirmatory quantities and IIA with every alignment frozen. The guard refuses to open EXT without the completion markers of 04 and 05, in smoke mode, or after the pass has been closed.

In [ ]:
# ============================ CELL 19 · EXT ============================
if STAGE == "EXT":
    if not I_CONFIRM_THE_SINGLE_EXT_PASS: raise PermissionError("EXT can be loaded for ONE pass. Set I_CONFIRM_THE_SINGLE_EXT_PASS = True in Cell 2 only when 04 and 05 are finished and committed.")
    S04 = json.load(open(REP("04_summary.json"))); CH = causal.verify_loc_choices(REP("04_loc_choices.json"))
    guard = data.FoldGuard("final_ext", prerequisites=[REP("04_confA_done.json"), REP("05_confB_done.json")])
    FB_E, BAT_E, INFO_E = open_fold("EXT", seed=C.CFG.SEED + 3000, label="EXT", guard=guard)
    onset_rows = metrics.row_onset(FB_E["store"], FB_E["onset"]); EXT = dict(core_commit=CORE_COMMIT, pairs=INFO_E, performance={}, replication={}, iia={})
    for seed in SEEDS:                                              # (i) frozen models, MIMIC statistics and VAL calibration applied as they are
        for arm in ARMS:
            nm = f"{arm}_s{seed}"; r = runs.external_report(B, nm, FB_E["store"], onset_rows, hours=False); EXT["performance"][nm] = r["all"]
            print(runs.line(nm, r["all"]))
    run_all(BAT_E, "EXT")
    for seed in SEEDS:                                              # (ii) replication with everything frozen
        for arm in ARMS:
            nm = f"{arm}_s{seed}"; e = BAT_E.eff(nm); rep = {}
            for part, key in (("priority", "up"), ("arousal", "arousal")):
                for s_ in S04["decisions"][nm][part]:
                    tr = S04["decisions"][nm][part][s_].get("trained", {}); heads, signs = tr.get("heads", []), tr.get("signs", []); ef = e[key][s_]
                    ok = [bool(np.sign(ef["mean"].ravel()[h]) == sg and (ef["lo"].ravel()[h] > 0 or ef["hi"].ravel()[h] < 0)) for h, sg in zip(heads, signs)]
                    rep[f"{part}/{s_}"] = dict(confirmed_on_CONF_A=len(heads), replicated_on_EXT=int(sum(ok)))
            raw = BAT_E.raw(nm); hm, _ = carried(nm); sgn = np.array(CH["beta_sign"][nm], float)
            EXT["iia"][nm] = {part: causal.iia_table(causal.iia(raw, BAT_E.P[part], sgn, part, floor=S04["iia_floor_constant_models"]), BAT_E.P[part], hm) for part in ("up", "arousal")}
            same = {k_: bool(np.sign(EXT["iia"][nm][p_][k_]["mean"]) == np.sign(S04["iia"][nm][p_][k_]["mean"])) for p_, k_ in (("up", "M1_minus_M0"), ("arousal", "M2_minus_M0"), ("arousal", "M2_minus_M1"))}
            EXT["replication"][nm] = dict(heads=rep, iia_difference_same_sign_as_CONF_A=same)
            print(f"{nm}: heads confirmed on CONF-A that replicate on EXT (same sign, CI excluding 0): {rep} | IIA differences keep their sign: {same}")
    p_ = runs.save_json(EXT, "04_ext_final.json"); data.ext_ledger_close(p_)
    print("EXT pass complete and closed in the ledger. EXT cannot be loaded again.")

## What to send back

- Stage LOC: the printed output of Cells 3–11. Then commit `Docs/LOC_CHOICES.md`.
- Stage CONF-A: the printed output of Cells 12–17, or `reports/04_summary.json`.
- Stage EXT: the printed output of Cell 19, or `reports/04_ext_final.json`.

Nothing here was executed before hand-over. Run it with `SMOKE_TEST = True` first (stages LOC and CONF_A; the smoke run never loads CONF-A) and send me any traceback.